# 02 — CKA Analysis: Measuring Representational Similarity

**Goal**: Understand how CKA (Centered Kernel Alignment) works and use it to monitor information preservation during transformer → Mamba conversion.

CKA measures how similar two neural network layer representations are, invariant to orthogonal transformations and isotropic scaling. We use it as our primary metric for conversion quality.

**Key threshold**: CKA < 0.75 signals excessive information loss and triggers investigation.

---

**References**:
- Kornblith et al., "Similarity of Neural Network Representations Revisited" (ICML 2019)
- Nguyen et al., "Do Wide Neural Networks Really Need to be Wide?" (AAAI 2021)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from distill.cka import (
    linear_cka, rbf_cka, minibatch_cka,
    cka_permutation_test, MinibatchCKAAccumulator,
    compute_layerwise_cka,
)

torch.manual_seed(42)
np.random.seed(42)
print("CKA module loaded successfully")

## 1. CKA on Synthetic Data — Building Intuition

Let's verify CKA behaves as expected:
- Identical representations → CKA = 1.0
- Orthogonal transform of same data → CKA = 1.0 (invariance!)
- Random noise → CKA ≈ 0.0
- Partial corruption → CKA between 0 and 1

In [ ]:
n_samples, d = 256, 64

X = torch.randn(n_samples, d)

# 1. Identical
cka_identical = linear_cka(X, X).item()
print(f"Identical:       CKA = {cka_identical:.4f}")

# 2. Orthogonal transform (should be ~1.0)
Q, _ = torch.linalg.qr(torch.randn(d, d))
Y_ortho = X @ Q
cka_ortho = linear_cka(X, Y_ortho).item()
print(f"Orthogonal:      CKA = {cka_ortho:.4f}")

# 3. Scaled (should be 1.0 — CKA is scale-invariant)
Y_scaled = X * 5.0
cka_scaled = linear_cka(X, Y_scaled).item()
print(f"Scaled (5x):     CKA = {cka_scaled:.4f}")

# 4. Random noise (should be ~0)
Y_noise = torch.randn(n_samples, d)
cka_noise = linear_cka(X, Y_noise).item()
print(f"Random noise:    CKA = {cka_noise:.4f}")

# 5. Partial corruption (varying noise levels)
print("\n--- Corruption sweep ---")
noise_levels = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
cka_scores = []
for sigma in noise_levels:
    Y_corrupt = X + torch.randn_like(X) * sigma
    score = linear_cka(X, Y_corrupt).item()
    cka_scores.append(score)
    print(f"  noise={sigma:.1f}: CKA = {score:.4f}")

# 6. Different dimensions (CKA handles d_x != d_y)
Y_small = X[:, :32]  # Project to half dims
cka_dim = linear_cka(X, Y_small).item()
print(f"\nDim mismatch ({d}→32): CKA = {cka_dim:.4f}")

In [ ]:
# Plot CKA vs noise level
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(noise_levels, cka_scores, "o-", color="#2196F3", linewidth=2, markersize=8)
ax.axhline(y=0.75, color="red", linestyle="--", alpha=0.7, label="Threshold (0.75)")
ax.fill_between(noise_levels, 0.75, 1.0, alpha=0.1, color="green", label="Acceptable zone")
ax.fill_between(noise_levels, 0, 0.75, alpha=0.1, color="red", label="Danger zone")
ax.set_xlabel("Noise Standard Deviation (sigma)", fontsize=12)
ax.set_ylabel("Linear CKA", fontsize=12)
ax.set_title("CKA Sensitivity to Gaussian Noise", fontsize=14)
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../results/cka_noise_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Linear CKA vs RBF CKA

Linear CKA is fast and captures linear relationships. RBF CKA captures nonlinear structure but is O(n^2). Let's compare them.

In [ ]:
# Compare linear vs RBF CKA across noise levels
lin_scores = []
rbf_scores = []

for sigma in noise_levels:
    Y_corrupt = X + torch.randn_like(X) * sigma
    lin_scores.append(linear_cka(X, Y_corrupt).item())
    rbf_scores.append(rbf_cka(X, Y_corrupt).item())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(noise_levels, lin_scores, "o-", label="Linear CKA", linewidth=2)
ax.plot(noise_levels, rbf_scores, "s--", label="RBF CKA", linewidth=2)
ax.axhline(y=0.75, color="red", linestyle=":", alpha=0.5, label="Threshold")
ax.set_xlabel("Noise sigma")
ax.set_ylabel("CKA Score")
ax.set_title("Linear vs RBF CKA")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nRBF CKA is more sensitive to noise (decays faster)."
      "\nWe use Linear CKA for monitoring during conversion (faster, smoother).")

## 3. Mini-batch CKA

For large activation matrices (millions of tokens), we can't hold everything in memory. Mini-batch CKA accumulates covariance statistics incrementally. Let's verify it matches the full computation.

In [ ]:
n_large = 2048
X_large = torch.randn(n_large, 128)
Y_large = X_large @ torch.randn(128, 96) + torch.randn(n_large, 96) * 0.3

# Full CKA
full_cka = linear_cka(X_large, Y_large).item()

# Mini-batch CKA with different batch sizes
for bs in [32, 64, 128, 256, 512]:
    mb_cka = minibatch_cka(X_large, Y_large, batch_size=bs)
    diff = abs(full_cka - mb_cka)
    status = "MATCH" if diff < 1e-4 else "DRIFT"
    print(f"  batch_size={bs:4d}: CKA={mb_cka:.6f} (diff={diff:.2e}) [{status}]")

print(f"\n  Full CKA:        {full_cka:.6f}")
print(f"\nMini-batch CKA is exact for linear kernel (covariance is additive).")

## 4. Permutation Test

Is a given CKA score statistically significant? We shuffle one matrix and compute the null distribution.

In [ ]:
# Test with correlated data (should be significant)
Y_corr = X + torch.randn_like(X) * 0.5
result = cka_permutation_test(X, Y_corr, n_permutations=500)

print("=== Permutation Test (correlated data) ===")
print(f"  Observed CKA: {result['observed_cka']:.4f}")
print(f"  p-value:      {result['p_value']:.4f}")
print(f"  Null mean:    {result['null_mean']:.4f}")
print(f"  Null std:     {result['null_std']:.4f}")
print(f"  Significant:  {'Yes (p < 0.05)' if result['p_value'] < 0.05 else 'No'}")

print()

# Test with uncorrelated data (should NOT be significant)
Y_rand = torch.randn_like(X)
result2 = cka_permutation_test(X, Y_rand, n_permutations=500)

print("=== Permutation Test (random data) ===")
print(f"  Observed CKA: {result2['observed_cka']:.4f}")
print(f"  p-value:      {result2['p_value']:.4f}")
print(f"  Null mean:    {result2['null_mean']:.4f}")
print(f"  Significant:  {'Yes (p < 0.05)' if result2['p_value'] < 0.05 else 'No'}")

## 5. Simulated Layer-wise CKA Heatmap

During distillation, we compute CKA between every teacher layer and every student layer. This reveals which teacher layers map best to which student layers — and where information is lost.

Here we simulate this with synthetic "activations" from a 32-layer teacher and 24-layer student.

In [ ]:
# Simulate teacher (32 layers, d=4096→projected) and student (24 layers, d=1024)
n_tokens = 512
n_teacher_layers = 32
n_student_layers = 24
d_teacher = 128   # Simulated projection dim
d_student = 64

# Create base signal that evolves through layers
base = torch.randn(n_tokens, d_teacher)
teacher_acts = {}
for i in range(n_teacher_layers):
    # Each layer transforms the representation progressively
    W = torch.eye(d_teacher) + torch.randn(d_teacher, d_teacher) * 0.05
    base = base @ W + torch.randn(n_tokens, d_teacher) * 0.1
    teacher_acts[f"teacher_layer_{i:02d}"] = base.clone()

# Student layers: initialized from teacher with noise (simulating conversion)
student_acts = {}
teacher_per_student = n_teacher_layers / n_student_layers
for j in range(n_student_layers):
    t_idx = int(j * teacher_per_student)
    t_act = teacher_acts[f"teacher_layer_{t_idx:02d}"]
    # Project to student dim + conversion noise
    proj = torch.randn(d_teacher, d_student) * 0.1
    s_act = t_act @ proj + torch.randn(n_tokens, d_student) * 0.3
    student_acts[f"student_layer_{j:02d}"] = s_act

# Compute full CKA heatmap
heatmap = compute_layerwise_cka(teacher_acts, student_acts)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    heatmap.scores,
    ax=ax,
    cmap="RdYlGn",
    vmin=0, vmax=1,
    xticklabels=[f"S{i}" for i in range(n_student_layers)],
    yticklabels=[f"T{i}" for i in range(n_teacher_layers)],
    cbar_kws={"label": "CKA"},
)
ax.set_xlabel("Student Layers", fontsize=12)
ax.set_ylabel("Teacher Layers", fontsize=12)
ax.set_title("Layer-wise CKA Heatmap (Teacher × Student)", fontsize=14)

# Mark the diagonal mapping
for j in range(n_student_layers):
    t_idx = int(j * teacher_per_student)
    ax.plot(j + 0.5, t_idx + 0.5, "ko", markersize=4)

plt.tight_layout()
plt.savefig("../results/cka_heatmap_simulated.png", dpi=150, bbox_inches="tight")
plt.show()

# Report diagonal CKA scores (the ones that matter for conversion)
diag_scores = []
for j in range(n_student_layers):
    t_idx = int(j * teacher_per_student)
    score = heatmap.scores[t_idx, j]
    diag_scores.append(score)

print(f"\nDiagonal CKA (mapped pairs):")
print(f"  Mean: {np.mean(diag_scores):.4f}")
print(f"  Min:  {np.min(diag_scores):.4f} (layer {np.argmin(diag_scores)})")
print(f"  Below threshold: {sum(1 for s in diag_scores if s < 0.75)}/{n_student_layers}")

## Key Takeaways

1. **Linear CKA** is our primary metric — fast, exact in mini-batch mode, and smooth
2. **CKA < 0.75** is our danger threshold for block conversion quality
3. **Mini-batch CKA** gives exact results for linear kernel (use it for large-scale monitoring)
4. **Permutation testing** confirms whether similarity is statistically significant
5. **Layer-wise heatmaps** reveal the mapping structure between teacher and student

Next: Use these tools to monitor actual attention → SSM conversion in notebook 03.